# 03 — Regresioni polinomial, Ridge dhe Lasso

        **Pyetja udhëheqëse:** Kur bëhet një model fleksibël tepër fleksibël?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të dallohet nënpërshtatja nga mbipërshtatja.
- Të ndërtohet matrica polinomiale.
- Të kuptohet regularizimi Ridge/Lasso.
- Të interpretohet kompromisi bias–variance.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("03 — Regresioni polinomial, Ridge dhe Lasso")
print("Seed:", RANDOM_STATE)

# 1. Sensor jolinear — CORE

Një sensor real mund të ketë përgjigje jolineare. Modeli polinomial
$$
\hat y=\beta_0+\beta_1x+\cdots+\beta_dx^d
$$
është ende linear në parametra, por kompleksiteti rritet me shkallën $d$.

In [ ]:
x = np.linspace(-2.5, 2.5, 70)
y_true = 1.0 + 0.7*x - 0.25*x**2 + 0.08*x**3
y = y_true + rng.normal(0, 0.7, len(x))

fig, ax = plt.subplots()
ax.scatter(x, y, s=18, label="matje")
ax.plot(x, y_true, label="përgjigjja reale")
ax.legend(); ax.set_xlabel("Sinjali hyrës"); ax.set_ylabel("Përgjigjja")
ax.set_title("Kalibrim jolinear")
plt.show()

# 2. Nënpërshtatje → përshtatje e mirë → mbipërshtatje — CORE

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

degrees = [1, 3, 14]
grid = np.linspace(x.min(), x.max(), 400)

fig, ax = plt.subplots()
ax.scatter(x, y, s=16, label="të dhëna")
for d in degrees:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(x[:,None], y)
    ax.plot(grid, m.predict(grid[:,None]), label=f"shkalla {d}")
ax.set_ylim(y.min()-3, y.max()+3)
ax.legend(); ax.set_title("Tre regjime të kompleksitetit")
plt.show()

**Interpretim.** Shkalla 1 ka bias të madh; shkalla 14 ndjek edhe luhatjet e rastësishme. Modeli i mirë duhet të kapë strukturën e qëndrueshme, jo çdo pikë.

# 3. Gabimi trajnim–validim — CORE

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

Xtr, Xva, ytr, yva = train_test_split(x[:,None], y, test_size=0.35, random_state=RANDOM_STATE)
deg_grid = range(1, 16)
tr_err, va_err = [], []
for d in deg_grid:
    m = make_pipeline(PolynomialFeatures(d), LinearRegression()).fit(Xtr, ytr)
    tr_err.append(mean_squared_error(ytr, m.predict(Xtr))**0.5)
    va_err.append(mean_squared_error(yva, m.predict(Xva))**0.5)

fig, ax = plt.subplots()
ax.plot(list(deg_grid), tr_err, marker="o", label="trajnim")
ax.plot(list(deg_grid), va_err, marker="o", label="validim")
ax.set_xlabel("Shkalla polinomiale"); ax.set_ylabel("RMSE")
ax.legend(); ax.set_title("Kompleksiteti dhe përgjithësimi")
plt.show()

# 4. Ridge: kontroll i amplitudës së koeficientëve — CORE

Ridge minimizon
$$
\|y-X\beta\|_2^2+\lambda\|\beta\|_2^2.
$$
Termi i dytë penalizon koeficientë shumë të mëdhenj dhe e bën zgjidhjen më të qëndrueshme.

In [ ]:
from sklearn.linear_model import Ridge, Lasso
alphas = np.logspace(-5, 3, 40)
degree = 10
P = PolynomialFeatures(degree, include_bias=False)
Xp = P.fit_transform(x[:,None])

ridge_norms, lasso_nonzero = [], []
for a in alphas:
    r = Ridge(alpha=a).fit(Xp, y)
    ridge_norms.append(np.linalg.norm(r.coef_))
    l = Lasso(alpha=a, max_iter=20000).fit(Xp, y)
    lasso_nonzero.append(np.sum(np.abs(l.coef_) > 1e-8))

fig, ax = plt.subplots()
ax.semilogx(alphas, ridge_norms, marker=".")
ax.set_xlabel("α"); ax.set_ylabel("||β||₂")
ax.set_title("Ridge zvogëlon amplitudën e koeficientëve")
plt.show()

# 5. Lasso dhe rrallësia — CORE

In [ ]:
fig, ax = plt.subplots()
ax.semilogx(alphas, lasso_nonzero, marker=".")
ax.set_xlabel("α"); ax.set_ylabel("Numri i koeficientëve jo-zero")
ax.set_title("Lasso mund të zerojë koeficientë")
plt.show()

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Regularizimi dhe parashikimi — EXTENDED

In [ ]:
models = {
    "pa regularizim": LinearRegression(),
    "Ridge α=1": Ridge(alpha=1.0),
    "Lasso α=0.02": Lasso(alpha=0.02, max_iter=20000)
}
fig, ax = plt.subplots()
ax.scatter(x, y, s=14)
for name, est in models.items():
    pipe = make_pipeline(PolynomialFeatures(10), est).fit(x[:,None], y)
    ax.plot(grid, pipe.predict(grid[:,None]), label=name)
ax.set_ylim(y.min()-3, y.max()+3); ax.legend()
ax.set_title("Efekti i regularizimit")
plt.show()

# 7. Aplikim jo-fizik: konsum energjie — EXTENDED

In [ ]:
T = np.linspace(-5, 35, 120)
energy = 40 + 0.08*(T-20)**2 + rng.normal(0, 2.5, len(T))
m = make_pipeline(PolynomialFeatures(2), Ridge(alpha=0.1)).fit(T[:,None], energy)
fig, ax = plt.subplots()
ax.scatter(T, energy, s=12)
ax.plot(T, m.predict(T[:,None]))
ax.set_xlabel("Temperatura e jashtme [°C]"); ax.set_ylabel("Konsum relativ")
ax.set_title("Shembull jashtë Fizikës: energjia e ndërtesës")
plt.show()

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Gjeni shkallën me RMSE minimal në validim.
2. Krahasoni koeficientët e Ridge për dy vlera të α.
3. Shpjegoni pse Lasso mund të përdoret për përzgjedhje veçorish.
4. Testoni modelin jashtë intervalit të të dhënave dhe diskutoni ekstrapolimin.

        ## B. Ushtrim i plotë
        Ndërtoni një studim Monte Carlo të gabimit trajnim/provë për shkallë 1–12 dhe tregoni shpërndarjen e shkallës së zgjedhur.

        ## C. Playground
        Ndryshoni zhurmën, shkallën polinomiale dhe α.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.